In [2]:
%pip install spacy

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached smart_open-8.0.1-py3-none-any.whl.metadata (24 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached wrapt-2.4.0-cp314-cp314-win_amd64.whl.metadata (7.6 kB)
   ---------------------------------------- 0.0/15.2 MB ? eta -:--:--
   --------------- ------------------------ 5.8/15.2 MB 43.9 MB/s eta 0:00:01
   ---------------------- ----------------- 8.4/15.2 MB 40.1 MB/s eta 0:00:01
   ----------------------- ---------------- 8.9/15.2 MB 16.7 MB/s eta 0:00:01
   ----------------------------------- ---- 13.4/15.2 MB 17.6 MB/s eta 0:00:01
   ---------------------------------------- 15.2/15.2 MB 18.2 MB/s  0:00:00
   ------------


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
import spacy
import lightgbm as lgb

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

In [2]:
dataset = pd.read_csv("../data/processed/processed_comments.csv")

cleaned_dataset = dataset.dropna(
    subset=["clean_comment", "category"]
).copy()

X_cleaned = cleaned_dataset["clean_comment"]
y_cleaned = cleaned_dataset["category"]

print(cleaned_dataset.shape)
print(y_cleaned.value_counts())

(36662, 2)
category
 1    15770
 0    12644
-1     8248
Name: count, dtype: int64


In [3]:
X_train_cleaned, X_test_cleaned, y_train_cleaned, y_test_cleaned = train_test_split(
    X_cleaned,
    y_cleaned,
    test_size=0.2,
    random_state=42,
    stratify=y_cleaned
)

print("Train:", X_train_cleaned.shape)
print("Test:", X_test_cleaned.shape)

Train: (29329,)
Test: (7333,)


In [7]:
%pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --- ------------------------------------ 1.0/12.8 MB 16.4 MB/s eta 0:00:01
     --------------------------------------  12.6/12.8 MB 49.5 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 45.3 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
nlp = spacy.load("en_core_web_sm")

In [5]:
POS_TAGS = [
    "ADJ", "ADP", "ADV", "AUX", "CCONJ",
    "DET", "INTJ", "NOUN", "NUM", "PART",
    "PRON", "PROPN", "PUNCT", "SCONJ",
    "SYM", "VERB", "X"
]

def extract_custom_features(text):

    doc = nlp(text)

    tokens = list(doc)

    word_count = len(tokens)
    comment_length = len(text)

    avg_word_length = (
        sum(len(token.text) for token in tokens) / word_count
        if word_count > 0 else 0
    )

    unique_word_count = len(
        set(token.text for token in tokens)
    )

    lexical_diversity = (
        unique_word_count / word_count
        if word_count > 0 else 0
    )

    features = {
        "comment_length": comment_length,
        "word_count": word_count,
        "avg_word_length": avg_word_length,
        "unique_word_count": unique_word_count,
        "lexical_diversity": lexical_diversity
    }

    pos_counts = {
        tag: 0
        for tag in POS_TAGS
    }

    for token in tokens:
        if token.pos_ in pos_counts:
            pos_counts[token.pos_] += 1

    for tag in POS_TAGS:
        features[f"pos_{tag}"] = (
            pos_counts[tag] / word_count
            if word_count > 0 else 0
        )

    return features

In [6]:
train_custom_features = pd.DataFrame(
    [
        extract_custom_features(text)
        for text in X_train_cleaned
    ]
)

test_custom_features = pd.DataFrame(
    [
        extract_custom_features(text)
        for text in X_test_cleaned
    ]
)

print(train_custom_features.shape)
print(test_custom_features.shape)

train_custom_features.head()

(29329, 22)
(7333, 22)


,comment_length,word_count,avg_word_length,unique_word_count,lexical_diversity,pos_ADJ,pos_ADP,pos_ADV,pos_AUX,pos_CCONJ,...,pos_NOUN,pos_NUM,pos_PART,pos_PRON,pos_PROPN,pos_PUNCT,pos_SCONJ,pos_SYM,pos_VERB,pos_X
0,398,57,6.000000,43,0.754386,0.035088,0.017544,0.017544,0.035088,0.017544,...,0.543860,0.017544,0.000000,0.035088,0.017544,0.0,0.0,0.0,0.228070,0.017544
1,100,16,5.312500,15,0.937500,0.062500,0.000000,0.187500,0.062500,0.062500,...,0.312500,0.000000,0.000000,0.000000,0.187500,0.0,0.0,0.0,0.125000,0.000000
2,339,51,5.666667,49,0.960784,0.137255,0.000000,0.078431,0.000000,0.019608,...,0.411765,0.000000,0.019608,0.000000,0.098039,0.0,0.0,0.0,0.215686,0.000000
3,77,12,5.500000,12,1.000000,0.250000,0.000000,0.000000,0.083333,0.000000,...,0.333333,0.000000,0.000000,0.000000,0.083333,0.0,0.0,0.0,0.250000,0.000000
4,181,28,5.500000,27,0.964286,0.000000,0.035714,0.000000,0.000000,0.000000,...,0.107143,0.000000,0.000000,0.000000,0.785714,0.0,0.0,0.0,0.071429,0.000000


In [7]:
train_custom_features = train_custom_features.fillna(0)
test_custom_features = test_custom_features.fillna(0)

print(train_custom_features.isnull().sum().sum())
print(test_custom_features.isnull().sum().sum())

0
0


In [8]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=10000
)

X_train_tfidf = tfidf.fit_transform(
    X_train_cleaned
)

X_test_tfidf = tfidf.transform(
    X_test_cleaned
)

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(29329, 10000)
(7333, 10000)


In [9]:
train_custom_sparse = csr_matrix(
    train_custom_features.values
)

test_custom_sparse = csr_matrix(
    test_custom_features.values
)

X_train_combined = hstack(
    [
        X_train_tfidf,
        train_custom_sparse
    ]
).tocsr()

X_test_combined = hstack(
    [
        X_test_tfidf,
        test_custom_sparse
    ]
).tocsr()

print("Combined train:", X_train_combined.shape)
print("Combined test:", X_test_combined.shape)

Combined train: (29329, 10022)
Combined test: (7333, 10022)


In [11]:
model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=3,
    class_weight="balanced",

    learning_rate=0.11621904727844506,
    n_estimators=901,
    max_depth=10,

    reg_alpha=0.1,
    reg_lambda=0.1,

    random_state=42,
    verbosity=-1
)

print(model)

LGBMClassifier(class_weight='balanced', learning_rate=0.11621904727844506,
               max_depth=10, n_estimators=901, num_class=3,
               objective='multiclass', random_state=42, reg_alpha=0.1,
               reg_lambda=0.1, verbosity=-1)


In [12]:
model.fit(
    X_train_combined,
    y_train_cleaned
)

,max_depth,10
,learning_rate,0.11621904727844506
,n_estimators,901
,objective,'multiclass'
,class_weight,'balanced'
,reg_alpha,0.1
,reg_lambda,0.1
,random_state,42
,num_class,3
,verbosity,-1
,boosting_type,'gbdt'


In [13]:
y_pred = model.predict(
    X_test_combined
)

accuracy = accuracy_score(
    y_test_cleaned,
    y_pred
)

print("Test Accuracy:", accuracy)

print(
    classification_report(
        y_test_cleaned,
        y_pred
    )
)

Test Accuracy: 0.8641756443474703
              precision    recall  f1-score   support

          -1       0.80      0.76      0.78      1650
           0       0.86      0.95      0.90      2529
           1       0.90      0.85      0.87      3154

    accuracy                           0.86      7333
   macro avg       0.85      0.85      0.85      7333
weighted avg       0.86      0.86      0.86      7333

